# S02 - Première table Delta

## Objectif

Construire un flux Databricks reproductible:
1. Créer des données synthétiques
2. Construire un Dataframe Pyspark avec schema explicite
3. Enregistrer une table Delta source
4. Controler la table en SQL
5. Trasformer les données avec Pyspark
6. Enrgistrer une seconde table Delta

In [0]:
print('Test de la session spark')
spark.range(5).show()

Test de la session spark
+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



# Configuration 

Les tables sont enregistrées dans Unity Catalog

- Table source: `workspace.default.commandes_s02_source`
- Table transformée: `workspace.default.commandes_s02_transformees`

Le notebook supprime d'abord les anciennes tables afin de pouvoir être relancé intégralement

In [0]:
CATALOG = "workspace"
SCHEMA = "default"

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.commandes_s02_source"
TARGET_TABLE = f"{CATALOG}.{SCHEMA}.commandes_s02_transformees"

# Afficher le catalogue et le schéma courants
spark.sql("""
    SELECT
        current_catalog() AS catalog,
        current_schema() AS schema
""").show(truncate=False)

# Préparer les requêtes séparément pour éviter l’alerte de l’éditeur
drop_target_query = f"DROP TABLE IF EXISTS {TARGET_TABLE}"
drop_source_query = f"DROP TABLE IF EXISTS {SOURCE_TABLE}"

# Nettoyage pour rendre le notebook reproductible
spark.sql(drop_target_query)
spark.sql(drop_source_query)

print(f"Table source : {SOURCE_TABLE}")
print(f"Table cible  : {TARGET_TABLE}")

+---------+-------+
|catalog  |schema |
+---------+-------+
|workspace|default|
+---------+-------+

Table source : workspace.default.commandes_s02_source
Table cible  : workspace.default.commandes_s02_transformees


## 2. Données synthétiques

Le jeu de données représente huit commandes fictives.

Une ligne coorespond à une commande passée par un client pour un produit.
La ville peut être absente afin de tester le traitement des valeurs nulles

In [0]:
from datetime import date 
from decimal import Decimal

commandes = [
    (
        1001, 101, "ordinateur_portable", 1,
        Decimal("899.99"), date(2026, 8, 10),
        "livree", "Paris"
    ),
    (
        1002, 102, "clavier", 2,
        Decimal("25.00"), date(2026, 8, 10),
        "livree", "Lyon"
    ),
    (
        1003, 101, "souris", 4,
        Decimal("15.00"), date(2026, 8, 11),
        "en_preparation", "Lyon"
    ),
    (
        1004, 103, "ecran", 1,
        Decimal("120.00"), date(2026, 8, 11),
        "livree", "Marseille"
    ),
    (
        1005, 104, "casque", 3,
        Decimal("25.00"), date(2026, 8, 12),
        "livree", None
    ),
    (
        1006, 105, "tablette", 2,
        Decimal("350.00"), date(2026, 8, 12),
        "annulee", "Paris"
    ),
    (
        1007, 106, "smartphone", 1,
        Decimal("1300.00"), date(2026, 8, 13),
        "livree", "Lille"
    ),
    (
        1008, 107, "cable_usb", 5,
        Decimal("9.00"), date(2026, 8, 13),
        "en_preparation", "Lyon"
    )
]

print("Nombre de commandes préparées :", len(commandes))


Nombre de commandes préparées : 8


## 3. Contructoin du Dataframe

Le schéma est défini explicitement pour eviter que Spark dévine les types.

La colonne 'ville' est nullable. Les autres colonnes obligatoire ne le sont pas.

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DecimalType,
    DateType
)

schema_commandes = StructType([
    StructField('commande_id', IntegerType(), False),
    StructField('client_id', IntegerType(), False),
    StructField('produit', StringType(), False),
    StructField('quantite', IntegerType(), False),
    StructField('prix_unitaire', DecimalType(10, 2), False),
    StructField('date_commande', DateType(), False),
    StructField('statut', StringType(), False),
    StructField('ville', StringType(), True)
])

df_commandes = spark.createDataFrame(data=commandes, schema=schema_commandes)

In [0]:
df_commandes.printSchema()
display(df_commandes)

root
 |-- commande_id: integer (nullable = false)
 |-- client_id: integer (nullable = false)
 |-- produit: string (nullable = false)
 |-- quantite: integer (nullable = false)
 |-- prix_unitaire: decimal(10,2) (nullable = false)
 |-- date_commande: date (nullable = false)
 |-- statut: string (nullable = false)
 |-- ville: string (nullable = true)



commande_id,client_id,produit,quantite,prix_unitaire,date_commande,statut,ville
1001,101,ordinateur_portable,1,899.99,2026-08-10,livree,Paris
1002,102,clavier,2,25.00,2026-08-10,livree,Lyon
1003,101,souris,4,15.00,2026-08-11,en_preparation,Lyon
1004,103,ecran,1,120.00,2026-08-11,livree,Marseille
1005,104,casque,3,25.00,2026-08-12,livree,null
1006,105,tablette,2,350.00,2026-08-12,annulee,Paris
1007,106,smartphone,1,1300.00,2026-08-13,livree,Lille
1008,107,cable_usb,5,9.00,2026-08-13,en_preparation,Lyon


In [0]:
assert df_commandes.count() == 8, "Le nombre de lignes dans la table source est incorrect"
assert len(df_commandes.columns) == 8, "Le nombre de colonnes dans la table source est incorrect"
assert df_commandes.schema == schema_commandes, "Le schéma de la table source est incorrect"

print('Dataframe valide: 8 Lignes et 8 colonnes.')

Dataframe valide: 8 Lignes et 8 colonnes.


## 4. Ecriture de la tacble Delta source 

Le DataFrame est enregistré comme table Delta managée dans unity catalog.

Le mode 'overwrite' permet de recréer la table avec le mêm résultat lors d'une nouvelle  execution

In [0]:
(df_commandes.write
.format('delta')
.mode('overwrite')
.option('overwriteschema', 'true')
.saveAsTable(SOURCE_TABLE)
)

In [0]:
assert spark.catalog.tableExists(SOURCE_TABLE)
assert spark.table(SOURCE_TABLE).count()==8

display(spark.table(SOURCE_TABLE))

print('La table delta source est disponible.')

commande_id,client_id,produit,quantite,prix_unitaire,date_commande,statut,ville
1001,101,ordinateur_portable,1,899.99,2026-08-10,livree,Paris
1002,102,clavier,2,25.00,2026-08-10,livree,Lyon
1003,101,souris,4,15.00,2026-08-11,en_preparation,Lyon
1004,103,ecran,1,120.00,2026-08-11,livree,Marseille
1005,104,casque,3,25.00,2026-08-12,livree,null
1006,105,tablette,2,350.00,2026-08-12,annulee,Paris
1007,106,smartphone,1,1300.00,2026-08-13,livree,Lille
1008,107,cable_usb,5,9.00,2026-08-13,en_preparation,Lyon


La table delta source est disponible.


In [0]:
%sql
SELECT COUNT(*) AS nombre_commande
FROM workspace.default.commandes_s02_source;

nombre_commande
8


In [0]:
%sql
SELECT 
 statut,
 count(*) as nombre_commande,
 sum(quantite) as nombre_articles,
 round(sum(quantite * prix_unitaire), 2) as montant_total

from workspace.default.commandes_s02_source
group by statut
order by montant_total

statut,nombre_commande,nombre_articles,montant_total
en_preparation,2,9,105.00
annulee,1,2,700.00
livree,5,8,2444.99


In [0]:
%sql
SELECT
    commande_id,
    client_id,
    produit,
    quantite,
    prix_unitaire,
    quantite * prix_unitaire as montant_commande
FROM workspace.default.commandes_s02_source
WHERE statut = 'livree'
    AND quantite * prix_unitaire >= 100
ORDER BY montant_commande DESC
    

commande_id,client_id,produit,quantite,prix_unitaire,montant_commande
1007,106,smartphone,1,1300.00,1300.00
1001,101,ordinateur_portable,1,899.99,899.99
1004,103,ecran,1,120.00,120.00


In [0]:
%sql
SELECT
    SUM(CASE WHEN commande_id IS NULL THEN 1 ELSE 0 END)
        AS commande_id_null,
    SUM(CASE WHEN client_id IS NULL THEN 1 ELSE 0 END)
        AS client_id_null,
    SUM(CASE WHEN produit IS NULL THEN 1 ELSE 0 END)
        AS produit_null,
    SUM(CASE WHEN quantite IS NULL THEN 1 ELSE 0 END)
        AS quantite_null,
    SUM(CASE WHEN prix_unitaire IS NULL THEN 1 ELSE 0 END)
        AS prix_unitaire_null,
    SUM(CASE WHEN date_commande IS NULL THEN 1 ELSE 0 END)
        AS date_commande_null,
    SUM(CASE WHEN statut IS NULL THEN 1 ELSE 0 END)
        AS statut_null,
    SUM(CASE WHEN ville IS NULL THEN 1 ELSE 0 END)
        AS ville_null
FROM workspace.default.commandes_s02_source;

commande_id_null,client_id_null,produit_null,quantite_null,prix_unitaire_null,date_commande_null,statut_null,ville_null
0,0,0,0,0,0,0,1


In [0]:
from pyspark.sql.functions import (
    col,
    coalesce,
    lit,
    round as spark_round,
    when
)

df_source = spark.table(SOURCE_TABLE)

df_transforme = (
    df_source
    .withColumn(
        'montant_commande',
        spark_round(
            col('quantite') * col('prix_unitaire'),
            2
        )
    )
    .withColumn(
        'ville_normalisee',
        coalesce(col('ville'), lit('INCONNUE'))
    )
    .withColumn(
        'priorite',
        when(col('montant_commande') >= 500,
                 lit('HAUTE')
                 ).otherwise(lit('NORMALE')
        )
    )
)

display(df_transforme)

commande_id,client_id,produit,quantite,prix_unitaire,date_commande,statut,ville,montant_commande,ville_normalisee,priorite
1001,101,ordinateur_portable,1,899.99,2026-08-10,livree,Paris,899.99,Paris,HAUTE
1002,102,clavier,2,25.00,2026-08-10,livree,Lyon,50.00,Lyon,NORMALE
1003,101,souris,4,15.00,2026-08-11,en_preparation,Lyon,60.00,Lyon,NORMALE
1004,103,ecran,1,120.00,2026-08-11,livree,Marseille,120.00,Marseille,NORMALE
1005,104,casque,3,25.00,2026-08-12,livree,null,75.00,INCONNUE,NORMALE
1006,105,tablette,2,350.00,2026-08-12,annulee,Paris,700.00,Paris,HAUTE
1007,106,smartphone,1,1300.00,2026-08-13,livree,Lille,1300.00,Lille,HAUTE
1008,107,cable_usb,5,9.00,2026-08-13,en_preparation,Lyon,45.00,Lyon,NORMALE


In [0]:
(
    df_transforme.write
    .format('delta')
    .mode('overwrite')
    .option('overwriteSchema', 'true')
    .saveAsTable(TARGET_TABLE)

)

print(f'Table créée : {TARGET_TABLE}')

Table créée : workspace.default.commandes_s02_transformees


In [0]:
df_cible = spark.table(TARGET_TABLE)

assert spark.catalog.tableExists(TARGET_TABLE)
assert df_cible.count() == 8
assert df_cible.filter(col('ville_normalisee').isNull()).count() == 0

print('Table transformée valide')

Table transformée valide


In [0]:
%sql
SELECT
    commande_id,
    produit,
    montant_commande,
    ville_normalisee,
    priorite
FROM workspace.default.commandes_s02_transformees
ORDER BY montant_commande DESC

commande_id,produit,montant_commande,ville_normalisee,priorite
1007,smartphone,1300.00,Lille,HAUTE
1001,ordinateur_portable,899.99,Paris,HAUTE
1006,tablette,700.00,Paris,HAUTE
1004,ecran,120.00,Marseille,NORMALE
1005,casque,75.00,INCONNUE,NORMALE
1003,souris,60.00,Lyon,NORMALE
1002,clavier,50.00,Lyon,NORMALE
1008,cable_usb,45.00,Lyon,NORMALE


In [0]:
nombre_source = spark.table(SOURCE_TABLE).count()
nombre_cible = spark.table(TARGET_TABLE).count()

assert nombre_source == 8
assert nombre_cible == 8

print('Pipeline terminé avec succès.')
print(f'Table source : {SOURCE_TABLE} - {nombre_source} lignes')
print(f'Table cible : {TARGET_TABLE} - {nombre_cible} lignes')

Pipeline terminé avec succès.
Table source : workspace.default.commandes_s02_source - 8 lignes
Table cible : workspace.default.commandes_s02_transformees - 8 lignes
